# 02 — Five-tier engine on the four synthetic showcases

Imports `backend.runtime.ValveGuardRuntime` and `model.pulseai_engine`.
This is the product, not a notebook rewrite.

| Showcase | Expected |
|---|---|
| stable_surveillance | `no_hvd`, completed, numeric BVF |
| stage2_hvd_candidate | `stage_2`, completed, functional-course warning |
| stage3_fail_closed | `stage_3`, abstained, no Tier 2 head |
| suspected_thrombosis | alternative cause, not an SVD label |


In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent):
    if (_candidate / "_paths.py").is_file():
        _nb = str(_candidate.resolve())
        if _nb not in sys.path:
            sys.path.insert(0, _nb)
        break
else:
    raise RuntimeError("Open this notebook from the ValveGuard repo (root or notebooks/).")

from _paths import ARTIFACT_DIR
from backend.runtime import ValveGuardRuntime
from backend.tests.fixtures import presentation_demo_payloads
from model.pulseai_engine import EngineInput
from model.tier1_varc3_rules import evaluate_varc3_hvd

runtime = ValveGuardRuntime.load(ARTIFACT_DIR)
payloads = presentation_demo_payloads()
rows = []
for name, payload in payloads.items():
    engine_input = EngineInput.from_mapping(payload)
    rules = evaluate_varc3_hvd(
        engine_input.baseline_echo,
        engine_input.current_echo,
        engine_input.clinical_context,
    )
    result = runtime.predict(payload)
    bvf = None
    if result.get("tier2_first_event"):
        bvf = result["tier2_first_event"]["horizon_risks"]["1_year"]["svd_bvf"]
    rows.append(
        {
            "showcase": name,
            "tier1": rules.candidate_stage.value,
            "engine_status": result["engine_status"],
            "intervention": result["tier5_surveillance"]["intervention_recommendation"],
            "bvf_1y_raw_or_none": bvf,
            "trajectory": result.get("risk_trajectory", {}).get("available"),
            "withheld": [item["output"] for item in result.get("withheld_outputs", [])],
        }
    )
    print(name, rows[-1])

by_name = {row["showcase"]: row for row in rows}
assert by_name["stable_surveillance"]["tier1"] == "no_hvd"
assert by_name["stable_surveillance"]["engine_status"] == "completed"
assert by_name["stage2_hvd_candidate"]["tier1"] == "stage_2"
assert by_name["stage3_fail_closed"]["tier1"] == "stage_3"
assert by_name["stage3_fail_closed"]["engine_status"] == "abstained"
assert all(row["intervention"] is None for row in rows)


In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib optional; skip plot")

stable = runtime.predict(payloads["stable_surveillance"])
traj = stable.get("risk_trajectory") or {}
if plt and traj.get("available"):
    series = {item["id"]: item for item in traj.get("series", [])}
    fig, ax = plt.subplots(figsize=(8, 4))
    for key, style in (
        ("uncalibrated_svd_bvf", "-"),
        ("uncalibrated_other_cause_death", "-"),
        ("event_free", "--"),
    ):
        item = series.get(key)
        if not item:
            continue
        xs = [pt["years"] for pt in item["points"]]
        ys = [pt["value"] for pt in item["points"]]
        ax.plot(xs, ys, style, label=item["label"])
    ax.set_xlabel("Years from landmark")
    ax.set_ylabel("Probability")
    ax.set_title("Technical CIF/survival from the fitted engine (not clinical evidence)")
    ax.legend()
    ax.set_ylim(0, 1)
    plt.show()
print(
    "NSVD withheld",
    [
        item
        for item in stable.get("withheld_outputs", [])
        if "non_structural" in item.get("output", "")
    ],
)
